In [3]:
import requests
import json
import threading
import time
from websocket import WebSocketApp
from datetime import datetime, timezone

#Выбор ликвидного рынка

url = 'https://gamma-api.polymarket.com/markets?active=true&limit=50&order=volume24hr&ascending=false'
markets = requests.get(url).json()

target_market = None
token_ids = []

for m in markets:
    tokens = m.get('clobTokenIds', [])
    if tokens and m.get('acceptingOrders'):
        print(f"РЫНОК: {m['question']}...")
        print(f" Volume: ${m['volume24hr']:,}")

        token_ids = tokens[:3]
        target_market = m
        break

if not target_market:
    print(" Нет подходящих рынков")
    exit()

print(f" Подписка на {len(token_ids)} токенов")

#парсим

class MarketParser:

    def __init__(self, asset_ids):
        self.asset_ids = asset_ids
        self.url = "wss://ws-subscriptions-clob.polymarket.com/ws/market"

        self.prices = {aid: 0.0 for aid in asset_ids}
        self.best_bids = {aid: 0.0 for aid in asset_ids}

        self.events = []

    def heartbeat(self, ws):
        while True:
            time.sleep(10)
            try:
                ws.send("PING")
            except:
                break

    def on_open(self, ws):
        print(" WS подключен")

        sub = {
            "type": "market",
            "assets_ids": list(self.asset_ids),
            "custom_feature_enabled": True
        }

        ws.send(json.dumps(sub))
        print(" Подписка отправлена")

        threading.Thread(target=self.heartbeat, args=(ws,), daemon=True).start()

    def on_message(self, ws, message):

       
        if message.strip() == "PONG":
            print("got PONG")
            return

        try:
            data = json.loads(message)
        except:
            return

        if isinstance(data, list):
            for item in data:
                self.process_event(item)
        else:
            self.process_event(data)


    def process_event(self, data):
        event_type = data.get("event_type")
        asset_id = data.get("asset_id")

        now = datetime.now(timezone.utc)

        
        if event_type == "book":
            asks = data.get("asks", [])
            bids = data.get("bids", [])

            best_ask = float(asks[0][0]) if asks else 0
            best_bid = float(bids[0][0]) if bids else 0

            self.prices[asset_id] = best_ask
            self.best_bids[asset_id] = best_bid

            event = {
                "type": "book",
                "timestamp": now,
                "asset_id": asset_id,
                "best_ask": best_ask,
                "best_bid": best_bid
            }

            self.events.append(event)
            print(event)

        
        elif event_type == "last_trade_price":
            event = {
                "type": "trade",
                "timestamp": now,
                "asset_id": asset_id,
                "price": float(data.get("price", 0)),
                "size": float(data.get("size", 0)),
                "side": data.get("side")
            }

            self.events.append(event)
            print(event)


    def run(self):
        while True:
            try:
                ws = WebSocketApp(
                    self.url,
                    on_open=self.on_open,
                    on_message=self.on_message
                )
                ws.run_forever()

            except Exception as e:
                print("Ошибка WS:", e)

            print(" Переподключение через 5 сек...")
            time.sleep(5)


sim = MarketParser(token_ids)

try:
    sim.run()
except KeyboardInterrupt:
    print(" остановлено")

РЫНОК: US x Iran ceasefire by April 7?...
 Volume: $59,746,993.148844026
 Подписка на 3 токенов
 WS подключен
📤 Подписка отправлена
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
 Переподключение через 5 сек...
 остановлено
